<a href="https://colab.research.google.com/github/bonsii2/Abugida-Robotics/blob/main/fireRescues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.9 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()   # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"bonsaabonsii","key":"feaf45a5a01bc89fa5c346c2d2b9184b"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d azimjaan21/fire-and-smoke-dataset-object-detection-yolo

Dataset URL: https://www.kaggle.com/datasets/azimjaan21/fire-and-smoke-dataset-object-detection-yolo
License(s): Community Data License Agreement - Permissive - Version 1.0
^C


In [5]:
!kaggle datasets download -d uttejkumarkandagatla/fall-detection-dataset
!unzip fall-detection-dataset.zip -d fall_dataset

Dataset URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
License(s): ODbL-1.0
  0% 0.00/49.7M [00:00<?, ?B/s]
100% 49.7M/49.7M [00:00<00:00, 1.31GB/s]
Archive:  fall-detection-dataset.zip
  inflating: fall_dataset/fall_dataset/images/train/fall001.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall002.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall003.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall004.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall005.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall006.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall007.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall008.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall009.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall010.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fall011.jpg  
  inflating: fall_dataset/fall_dataset/images/train/fa

In [6]:
!ls fall_dataset

fall_dataset


In [7]:
%%writefile /content/fall_dataset.yaml
path: /content/fall_dataset/fall_dataset
train: images/train
val: images/val

names:
  0: fall

Writing /content/fall_dataset.yaml


In [8]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [9]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov8s.pt")  # you can use yolov8n.pt if you want lighter

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [10]:
model.train(
    data="/content/fall_dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    name="fall_detector"
)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fall_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fall_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a89540a7230>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [11]:
best_model = YOLO("/content/runs/detect/fall_detector/weights/best.pt")

In [12]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1391.3±1023.5 MB/s, size: 64.1 KB)
val: Scanning /content/fall_dataset/fall_dataset/labels/val.cache... 111 images, 0 backgrounds, 41 corrupt: 100% ━━━━━━━━━━━━ 111/111 42.3Mit/s 0.0s
val: /content/fall_dataset/fall_dataset/images/val/fall049.jpg: ignoring corrupt image/label: Label class 2 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/fall_dataset/fall_dataset/images/val/fall058.jpg: ignoring corrupt image/label: Label class 2 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/fall_dataset/fall_dataset/images/val/not fallen001.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/fall_dataset/fall_dataset/images/val/not fallen002.jpg: ignoring corrupt i

In [15]:
best_model.export(format="onnx")

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/fall_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 139ms
Prepared 4 packages in 6.16s
Installed 4 packages in 382ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime-gpu==1.24.2
 + onnxslim==0.1.86

requirements: AutoUpdate success ✅ 7.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 9.2s, saved as '/content/runs/detect/fall_detector/weights/best.onnx' (42.7 MB)

Export complete (10.0s)
Results saved to /content/runs/detect/fall_detector/weights
Predict:         yolo predict task=detect model=/content/runs/detect/fall_detector/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/fall_detector/weights/best.onnx imgsz=640 data=/content/fall_dataset.yaml  
Visualize:       https://netron.app


'/content/runs/detect/fall_detector/weights/best.onnx'